# <u>Lab Two - Classification</u>

Authors:  Muskaan Mahes, Chloe Prowse, Aayush Dalal, Nino Castellano

In [1]:
# Necessary imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

## 1) <u> Data Preparation </u>

### Define and Prepare Class Variables

The Student Placement Dataset from Kaggle was collected and analyzed to determine whether students’ results were sufficient to obtain a job offer. The dataset contained over 50,000 records consisting of academic, technical, and soft-skill attributes that can influence the outcome of being placed or not. <b>(This is our primary Classification Task #1)</b> Therefore, the primary purpose of the dataset is to help students, educational institutions, career centers understand which factors are crucial for achieving a successful placement outcome. 

We previously used this data to analyze and explore the relationships between students’ key features and their placement outcome to identify important insights. We then built and evaluated two classification models (SVM and Logistic Regression) to predict a student's placement status based on various academic, technical, and personal attributes. Now our objective is to take it a step further by training and evaluating different classification models for our existing classification task, Placement Status, while also training/evaluating those same models on a different classification task we can come up with from this data.

For example, as a <b>Secondary Classification Task #2 </b> aligned with our broader goal of understanding the factors that contribute to long-term career success, we could classify a student’s likelihood of securing an internship based on their current academic, technical, and personal attributes. Predicting internship attainment is particularly meaningful because internships often serve as a critical stepping stone toward full-time employment. By modeling internship probability, we can evaluate where a student currently stands in their professional development journey and identify areas for improvement. Since even a single internship can significantly increase the likelihood of receiving a full-time offer later, this secondary task provides both practical and predictive value alongside our primary placement classification objective.

Summarizing Clasification Tasks:

- <b>Placement Outcome (Yes or No) </b>
- <b>Internship Placement (Yes or No) </b>

Together, these two classification tasks create a developmental pathway: predicting internship probability drives short-term skill adjustments that increase immediate opportunities, and those strengthened skills directly feed into improved placement outcomes, ultimately supporting long-term full-time career success.

In [20]:
#loading dataset
df = pd.read_csv("../Student Placement Dataset/full_dataset.csv")
print(df.shape)
df.head()

(50000, 15)


,Student_ID,Age,Gender,Degree,Branch,CGPA,Internships,Projects,Coding_Skills,Communication_Skills,Aptitude_Test_Score,Soft_Skills_Rating,Certifications,Backlogs,Placement_Status
0,1048,22,Female,B.Tech,ECE,6.29,0,3,4,6,51,5,1,3,Not Placed
1,37820,20,Female,BCA,ECE,6.05,1,4,6,8,59,8,2,1,Not Placed
2,49668,22,Male,MCA,ME,7.22,1,4,6,6,58,6,2,2,Not Placed
3,19467,22,Male,MCA,ME,7.78,2,4,6,6,90,4,2,0,Placed
4,23094,20,Female,B.Tech,ME,7.63,1,4,6,5,79,6,2,0,Placed


### Describe Final Dataset

Using our previously preprocessed from earlier labs which has already been cleaned, and verified to contain no missing values; the data is ready preprocessing using encoding and scaling and then for model training and testing. The only additional step required is to create a new encoded target variable, <b>Internship Placement</b>, derived from the <b>Internships</b> feature. Students with at least one internship will be labeled <b>Yes</b>, while those with none will be labeled <b>No</b>.

In [21]:
#dropping ID Because it is not useful for the model
df = df.drop(columns=["Student_ID"], errors='ignore')

# Create Internship Placement binary target variable
df["Internship_Placement"] = np.where(df["Internships"] >= 1, "Yes", "No")

#identifying numeric and categorical columns
numeric_columns = df.select_dtypes(include=[np.number]).columns
categorical_columns = df.select_dtypes(include=['object']).columns

# Verify the identified columns
print("Numeric Columns:", len(numeric_columns))
print(numeric_columns)
print("Categorical Columns:", len(categorical_columns))
print(categorical_columns)

Numeric Columns: 10
Index(['Age', 'CGPA', 'Internships', 'Projects', 'Coding_Skills',
       'Communication_Skills', 'Aptitude_Test_Score', 'Soft_Skills_Rating',
       'Certifications', 'Backlogs'],
      dtype='str')
Categorical Columns: 5
Index(['Gender', 'Degree', 'Branch', 'Placement_Status',
       'Internship_Placement'],
      dtype='str')


C:\Users\SeanPC\AppData\Local\Temp\ipykernel_19324\639426551.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include=['object']).columns


In [22]:
print(df.shape)
df.head()

(50000, 15)


,Age,Gender,Degree,Branch,CGPA,Internships,Projects,Coding_Skills,Communication_Skills,Aptitude_Test_Score,Soft_Skills_Rating,Certifications,Backlogs,Placement_Status,Internship_Placement
0,22,Female,B.Tech,ECE,6.29,0,3,4,6,51,5,1,3,Not Placed,No
1,20,Female,BCA,ECE,6.05,1,4,6,8,59,8,2,1,Not Placed,Yes
2,22,Male,MCA,ME,7.22,1,4,6,6,58,6,2,2,Not Placed,Yes
3,22,Male,MCA,ME,7.78,2,4,6,6,90,4,2,0,Placed,Yes
4,20,Female,B.Tech,ME,7.63,1,4,6,5,79,6,2,0,Placed,Yes


In [ ]:
df['Placement_Status'].value_counts(normalize=True) * 100

Placement_Status
Not Placed    63.752
Placed        36.248
Name: proportion, dtype: float64

In [24]:
df['Internship_Placement'].value_counts(normalize=True) * 100

Internship_Placement
Yes    54.094
No     45.906
Name: proportion, dtype: float64

## 2) <u> Modeling and Evaluation </u>

### Choosing and Explaining Evaluation Metrics

For this project, precision will be used as the primary evaluation metric for both classification tasks: predicting internship placement and predicting full-time job placement. Precision measures the proportion of predicted positive outcomes that are actually correct. In this context, precision evaluates how often the model is correct when it predicts that a student will receive an internship or full-time job offer.

Precision is calculated as:

$$
Precision = \frac{TP}{TP + FP}
$$

where TP (True Positives) represents students who were correctly predicted to receive an internship or job offer, and FP (False Positives) represents students who were predicted to receive an offer but ultimately did not.

Precision is an appropriate evaluation metric for these classification tasks because it focuses on the reliability of positive predictions. A model with low precision would produce a large number of false positives, meaning that students may be predicted to receive internship or full-time job offers when they are not actually on track to secure those opportunities. If used by a career center to evaluate student readiness and career preparedness, these incorrect predictions could create misleading assessments of students’ current skills and professional development, making it more difficult to accurately identify students who may need additional support or skill development to improve their chances of obtaining an offer. This would reduce the usefulness of the model and could lead to misleading interpretations about student career outcomes.

Additionally, precision helps ensure that the students identified by the model as likely to receive internships or job offers are those who truly have a strong probability of achieving those outcomes. By prioritizing the correctness of positive predictions, the model provides more trustworthy insights into the characteristics and factors that contribute to successful internship and job placement.

Precision will therefore be used to evaluate both the Internship Placement classification model and the Full-Time Placement classification model, allowing for consistent evaluation across both prediction tasks while emphasizing the reliability of positive placement predictions.

### Method for Splitting Train and Test Sets

To evaluate the performance of the classification models, the dataset was divided into training and testing subsets using a 70/30 train-test split with stratified sampling. This method was applied to both classification tasks: predicting full-time placement status and predicting internship placement.

Stratified sampling was used during the train-test split to ensure that the proportion of classes in the training and testing datasets remains consistent with the original dataset. This is particularly important when class distributions are not perfectly balanced. For example, in the Full-Time Placement classification task, approximately 63.75% of students are labeled as "Not Placed" and 36.25% as "Placed". Similarly, in the Internship Placement classification task, approximately 54.09% of students have an internship placement and 45.91% do not.

Without stratification, random sampling could produce training or testing datasets where these class proportions differ significantly from the original dataset. Such imbalances could negatively affect model training or lead to unreliable evaluation results. By using stratified sampling, the class distributions are preserved in both the training and testing sets, ensuring that the models are trained and evaluated on data that accurately reflects the overall dataset.

A 70/30 split was chosen because it provides a good balance between training and evaluation data. The majority of the data is used for training the models, which helps them learn meaningful patterns, while still leaving a sufficiently large portion of the data for testing.

In [25]:
# Encoding Placement_Status (Task 1)
le_task1 = LabelEncoder()
df["Placement_Status"] = le_task1.fit_transform(df["Placement_Status"])

# Encoding Internship_Placement (Task 2)
le_task2 = LabelEncoder()
df["Internship_Placement"] = le_task2.fit_transform(df["Internship_Placement"])

In [26]:
df.head()

,Age,Gender,Degree,Branch,CGPA,Internships,Projects,Coding_Skills,Communication_Skills,Aptitude_Test_Score,Soft_Skills_Rating,Certifications,Backlogs,Placement_Status,Internship_Placement
0,22,Female,B.Tech,ECE,6.29,0,3,4,6,51,5,1,3,0,0
1,20,Female,BCA,ECE,6.05,1,4,6,8,59,8,2,1,0,1
2,22,Male,MCA,ME,7.22,1,4,6,6,58,6,2,2,0,1
3,22,Male,MCA,ME,7.78,2,4,6,6,90,4,2,0,1,1
4,20,Female,B.Tech,ME,7.63,1,4,6,5,79,6,2,0,1,1


In [27]:
df['Placement_Status'].value_counts(normalize=True) * 100

Placement_Status
0    63.752
1    36.248
Name: proportion, dtype: float64

In [28]:
df['Internship_Placement'].value_counts(normalize=True) * 100

Internship_Placement
1    54.094
0    45.906
Name: proportion, dtype: float64

In [29]:
# Getting Test and Train Split for Task 1: Placement Status Classification
X_task1 = df.drop(columns=["Placement_Status", "Internship_Placement"])
y_task1 = df["Placement_Status"]

# Identify feature types AFTER dropping targets
numeric_columns_task1 = X_task1.select_dtypes(include=np.number).columns
categorical_columns_task1 = X_task1.select_dtypes(include="object").columns

# Preprocessor for Task 1
preprocessor_task1 = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_columns_task1),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_columns_task1)
    ]
)

# Train Test Split
X_train_task1, X_test_task1, y_train_task1, y_test_task1 = train_test_split(
    X_task1,
    y_task1,
    test_size=0.3,
    random_state=42,
    stratify=y_task1
)

# Fit ONLY on train
X_train_task1 = preprocessor_task1.fit_transform(X_train_task1)
X_test_task1 = preprocessor_task1.transform(X_test_task1)

C:\Users\SeanPC\AppData\Local\Temp\ipykernel_19324\1607617089.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns_task1 = X_task1.select_dtypes(include="object").columns


In [30]:
X_task1.head()

,Age,Gender,Degree,Branch,CGPA,Internships,Projects,Coding_Skills,Communication_Skills,Aptitude_Test_Score,Soft_Skills_Rating,Certifications,Backlogs
0,22,Female,B.Tech,ECE,6.29,0,3,4,6,51,5,1,3
1,20,Female,BCA,ECE,6.05,1,4,6,8,59,8,2,1
2,22,Male,MCA,ME,7.22,1,4,6,6,58,6,2,2
3,22,Male,MCA,ME,7.78,2,4,6,6,90,4,2,0
4,20,Female,B.Tech,ME,7.63,1,4,6,5,79,6,2,0


In [8]:
# Getting Test and Train Data for Task 2: Internship Placement Classification
X_task2 = df.drop(columns=[
    "Internship_Placement", # target variable for Task 2
    "Internships",          # used to create target
    "Placement_Status"      # logically happens after internship
])

y_task2 = df["Internship_Placement"]

X_train_task2, X_test_task2, y_train_task2, y_test_task2 = train_test_split(
    X_task2, y_task2,
    test_size=0.3,
    random_state=42,
    stratify=y_task2
)

# IMPORTANT:
# Recreate column lists for Task 2 (since we dropped features)

numeric_columns_task2 = X_task2.select_dtypes(include=np.number).columns
categorical_columns_task2 = X_task2.select_dtypes(include="object").columns

preprocessor_task2 = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_columns_task2),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_columns_task2)
    ]
)

X_train_task2 = preprocessor_task2.fit_transform(X_train_task2)
X_test_task2 = preprocessor_task2.transform(X_test_task2)

C:\Users\SeanPC\AppData\Local\Temp\ipykernel_19324\497870786.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns_task2 = X_task2.select_dtypes(include="object").columns


In [32]:
X_task2.head()

,Age,Gender,Degree,Branch,CGPA,Projects,Coding_Skills,Communication_Skills,Aptitude_Test_Score,Soft_Skills_Rating,Certifications,Backlogs
0,22,Female,B.Tech,ECE,6.29,3,4,6,51,5,1,3
1,20,Female,BCA,ECE,6.05,4,6,8,59,8,2,1
2,22,Male,MCA,ME,7.22,4,6,6,58,6,2,2
3,22,Male,MCA,ME,7.78,4,6,6,90,4,2,0
4,20,Female,B.Tech,ME,7.63,4,6,5,79,6,2,0


In [33]:
# Final Ready-to-Use Data
print("TASK 1 Ready Shapes:")
print(X_train_task1.shape, X_test_task1.shape)

print("\nTASK 2 Ready Shapes:")
print(X_train_task2.shape, X_test_task2.shape)

TASK 1 Ready Shapes:
(35000, 18) (15000, 18)

TASK 2 Ready Shapes:
(35000, 17) (15000, 17)


<u>Note on how to Plug and Play Test and Train Sets into Models</u>

Task 1 Train and Test Sets (Placement Status):
X_train_task1, X_test_task1, y_train_task1, y_test_task1

Task 2 Train and Test Sets (Internship Placement):
X_train_task2, X_test_task2, y_train_task2, y_test_task2

Instructions on How to Use:

1. Initialize model
2. Fit on train
3. Predict on test
4. Evaluate

Example Classifier for Task 1:

Initialize

- rf = RandomForestClassifier(random_state=42)

Train

- rf.fit(X_train_task1, y_train_task1)

Predict on test set

- y_pred_task1 = rf.predict(X_test_task1)

Evaluate

- print("Accuracy:", accuracy_score(y_test_task1, y_pred_task1))
- print(classification_report(y_test_task1, y_pred_task1))



### Three Different Classification/Regression Models

### Analyze Model Results Using Choosen Evaluation Metric

### Discuss Advantages of Each Model on Each Classification/Regression Task

### Which Attributes are Most Important for each Classification Task

## 3) <u> Deployment </u>

### Usefulness of Model to Interested Parties 

## 4) <u> Exceptional Work </u>

### Additional Model or Hyperparameter Tuning Via Grid Search